# Ingesta de datos - Conversa AI
## Módulo de carga, validación y limpieza inicial

**Propósito:** Cargar archivos CSV/Excel (sintéticos o reales), detectar formato y encoding, corregir problemas comunes, y generar un DataFrame limpio con reporte de calidad.

**Autor:** Data Engineer  
**Fecha:** Mayo 2026

### 1. Setup y dependencias

In [21]:
import pandas as pd
import numpy as np
import chardet
import os
import re
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


### 2. Funciones auxiliares para detección de encoding y separador

In [22]:
def detect_encoding(file_path, sample_size=10000):
    """Detecta el encoding de un archivo usando chardet"""
    with open(file_path, 'rb') as f:
        raw_data = f.read(sample_size)
        result = chardet.detect(raw_data)
        return result['encoding'] if result['confidence'] > 0.7 else 'utf-8'

def detect_separator(file_path, encoding='utf-8', n_rows=5):
    """Detecta el separador más probable de un CSV (coma, punto y coma, tabulador)"""
    with open(file_path, 'r', encoding=encoding) as f:
        first_line = f.readline()
    candidates = [',', ';', '\t', '|']
    counts = {sep: first_line.count(sep) for sep in candidates}
    best_sep = max(counts, key=counts.get)
    return best_sep if counts[best_sep] > 0 else ','

def parse_file(file_path):
    """
    Detecta formato, encoding y separador, luego carga el archivo en un DataFrame.
    Retorna (df, metadata)
    """
    file_ext = Path(file_path).suffix.lower()
    
    # Detectar encoding
    encoding = detect_encoding(file_path)
    print(f"Encoding detectado: {encoding}")
    
    # Si es Excel
    if file_ext in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path, engine='openpyxl' if file_ext=='.xlsx' else 'xlrd')
        metadata = {'formato': 'Excel', 'encoding': encoding}
        return df, metadata
    
    # Si es CSV o texto
    elif file_ext == '.csv' or file_ext == '.txt':
        sep = detect_separator(file_path, encoding=encoding)
        print(f"Separador detectado: '{sep}'")
        try:
            df = pd.read_csv(file_path, encoding=encoding, sep=sep, dtype_backend='numpy_nullable')
        except Exception as e:
            # Fallback: intentar con engine='python' y sep automático
            print(f"Error con separador '{sep}': {e}. Intentando detección automática...")
            df = pd.read_csv(file_path, encoding=encoding, sep=None, engine='python')
        metadata = {'formato': 'CSV', 'encoding': encoding, 'separator': sep}
        return df, metadata
    else:
        raise ValueError(f"Formato de archivo no soportado: {file_ext}. Use .csv, .xlsx o .xls")

### 3. Función principal de ingesta

In [23]:
def ingest_file(file_path, expected_columns=None, date_col='fecha', frustration_col='nivel_frustracion'):
    """
    Carga y valida un archivo, realiza limpieza básica.
    
    Parámetros:
    - file_path: ruta del archivo
    - expected_columns: lista de columnas obligatorias (opcional)
    - date_col: nombre de la columna de fecha
    - frustration_col: nombre de columna de nivel de frustración
    
    Retorna:
    - df_clean: DataFrame limpio
    - quality_report: diccionario con métricas de calidad
    """
    quality_report = {
        'file_path': file_path,
        'status': 'ok',
        'errors': [],
        'warnings': [],
        'rows_initial': 0,
        'rows_final': 0,
        'columns_initial': [],
        'columns_final': [],
        'null_counts': {},
        'out_of_range_values': {}
    }
    
    # 1. Verificar existencia del archivo
    if not os.path.exists(file_path):
        quality_report['status'] = 'error'
        quality_report['errors'].append(f"Archivo no encontrado: {file_path}")
        return None, quality_report
    
    # 2. Parsear archivo
    try:
        df, metadata = parse_file(file_path)
        quality_report['rows_initial'] = len(df)
        quality_report['columns_initial'] = list(df.columns)
        print(f"Archivo cargado: {len(df)} filas, {len(df.columns)} columnas")
    except Exception as e:
        quality_report['status'] = 'error'
        quality_report['errors'].append(f"Error al parsear archivo: {str(e)}")
        return None, quality_report
    
    # 3. Validar columnas esperadas (si se proporcionan)
    if expected_columns:
        missing_cols = [col for col in expected_columns if col not in df.columns]
        if missing_cols:
            quality_report['errors'].append(f"Columnas faltantes: {missing_cols}")
            # No detenemos el proceso, pero reportamos
    
    # 4. Limpieza de nombres de columnas (strip, minúsculas, reemplazar espacios)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    
    # 5. Convertir columna de fecha
    if date_col in df.columns:
        try:
            df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
            null_dates = df[date_col].isna().sum()
            if null_dates > 0:
                quality_report['warnings'].append(f"{null_dates} fechas inválidas convertidas a NaT")
        except Exception as e:
            quality_report['warnings'].append(f"Error al convertir fechas: {str(e)}")
    else:
        quality_report['warnings'].append(f"Columna de fecha '{date_col}' no encontrada")
    
    # 6. Validar nivel de frustración (si existe)
    if frustration_col in df.columns:
        # Asegurar tipo numérico
        df[frustration_col] = pd.to_numeric(df[frustration_col], errors='coerce')
        out_of_range = df[(df[frustration_col] < 0) | (df[frustration_col] > 2)]
        if len(out_of_range) > 0:
            quality_report['out_of_range_values'][frustration_col] = len(out_of_range)
            # Corregir: limitar a [0,2] (opcional: podríamos eliminar)
            df.loc[df[frustration_col] < 0, frustration_col] = 0
            df.loc[df[frustration_col] > 2, frustration_col] = 2
            quality_report['warnings'].append(f"{len(out_of_range)} valores de frustración fuera de rango (0-2) corregidos")
    
    # 7. Eliminar filas completamente vacías
    before_drop = len(df)
    df = df.dropna(how='all')
    quality_report['rows_final'] = len(df)
    if before_drop - len(df) > 0:
        quality_report['warnings'].append(f"Eliminadas {before_drop - len(df)} filas completamente vacías")
    
    # 8. Reportar nulos por columna
    null_counts = df.isna().sum()
    quality_report['null_counts'] = null_counts[null_counts > 0].to_dict()
    
    # 9. Identificar y reportar tipos de datos incorrectos
    for col in df.columns:
        if df[col].dtype == 'object':
            # Chequeo rápido si la columna debería ser numérica (heurístico)
            if df[col].str.match(r'^-?\d+\.?\d*$').all():
                try:
                    df[col] = pd.to_numeric(df[col])
                    quality_report['warnings'].append(f"Columna '{col}' convertida a numérica")
                except:
                    pass
    
    quality_report['columns_final'] = list(df.columns)
    quality_report['metadata'] = metadata
    
    return df, quality_report

### 4. Aplicar ingesta

In [24]:
# encontrar ruta de corpus_v2_final.csv
potential_paths = [
	".data/data/raw/corpus_v2_final.csv",
	"../data/raw/corpus_v2_final.csv",
	"../../data/raw/corpus_v2_final.csv",
	"corpus_v2_final.csv"
]
for path in potential_paths:
	if os.path.exists(path):
		FILE_PATH = path
		print(f"Archivo encontrado: {FILE_PATH}")
		break

Archivo encontrado: ../data/raw/corpus_v2_final.csv


In [25]:
# Proceso de ingesta

# Columnas obligatorias según el proyecto
EXPECTED_COLS = [
    'session_id', 'turn_number', 'flow_name', 'usuario', 
    'fecha', 'intencion', 'nivel_frustracion', 
    'texto_espanol', 'texto_portugues', 'es_churn_risk', 'resolved'
]

# Ejecutar ingesta
df_limpio, report = ingest_file(FILE_PATH, expected_columns=EXPECTED_COLS)

# Mostrar reporte
print("\n=== REPORTE DE CALIDAD ===")
print(f"Estado: {report['status']}")
if report['errors']:
    print("ERRORES:")
    for e in report['errors']:
        print(f"  - {e}")
if report['warnings']:
    print("ADVERTENCIAS:")
    for w in report['warnings']:
        print(f"  - {w}")
print(f"Filas iniciales: {report['rows_initial']} -> finales: {report['rows_final']}")
print(f"Columnas finales: {report['columns_final']}")
print("Nulos por columna:", report['null_counts'])
print("Metadata:", report['metadata'])

Encoding detectado: UTF-8-SIG
Separador detectado: ','
Archivo cargado: 20001 filas, 11 columnas

=== REPORTE DE CALIDAD ===
Estado: ok
Filas iniciales: 20001 -> finales: 20001
Columnas finales: ['session_id', 'turn_number', 'flow_name', 'usuario', 'fecha', 'intencion', 'nivel_frustracion', 'texto_espanol', 'texto_portugues', 'es_churn_risk', 'resolved']
Nulos por columna: {}
Metadata: {'formato': 'CSV', 'encoding': 'UTF-8-SIG', 'separator': ','}


### 5. Definir valor esperados según el proyecto

In [26]:
# Definir valores esperados según el proyecto
EXPECTED_VALUES = {
    'flow_name': [
        'Acceso y Seguridad',
        'Gestion de Cuenta',      # Ojo: en tus datos aparece 'Gestión de Cuenta' (con tilde)
        'Soporte Técnico y Despacho',
        'Facturación y Cobros'
    ],
    'intencion': [
        'error_login',
        'cambio_plan',
        'logistica_envio',
        'problema_pago'
    ],
    'nivel_frustracion': [0, 1, 2],
    'turn_number': [1, 2, 3],     # según los datos sintéticos, máximo 3
    'es_churn_risk': [0, 1],
    'resolved': [0, 1]
}

def validate_domain(df, expected_values):
    """
    Valida que los valores de las columnas categóricas estén dentro de los permitidos.
    Retorna un diccionario con los valores no válidos encontrados.
    """
    invalid_report = {}
    for col, allowed in expected_values.items():
        if col not in df.columns:
            invalid_report[col] = f"Columna no existe"
            continue
        # Obtener valores únicos que NO están en la lista permitida
        invalid_vals = df[~df[col].isin(allowed)][col].unique()
        if len(invalid_vals) > 0:
            invalid_report[col] = list(invalid_vals)
    return invalid_report

# Ejecutar validación (después de tener df_limpio)
invalid = validate_domain(df_limpio, EXPECTED_VALUES)

if invalid:
    print("⚠️ Se encontraron valores no válidos en las columnas:")
    for col, vals in invalid.items():
        print(f"  - {col}: {vals}")
    # Opcional: decidir si eliminar esas filas o dejarlas como advertencia
else:
    print("✅ Todas las columnas categóricas tienen valores permitidos.")

⚠️ Se encontraron valores no válidos en las columnas:
  - flow_name: ['Gestión de Cuenta']


#### ℹ️ Hallazgos durante la ingesta

### Inconsistencia en `flow_name`
- **Problema:** El archivo contiene `'Gestión de Cuenta'` (con tilde en la 'o'), mientras que el dominio esperado es `'Gestion de Cuenta'` (sin tilde).
- **Decisión:** Se normalizó automáticamente reemplazando `'Gestión de Cuenta'` → `'Gestion de Cuenta'`.
- **Impacto:** 3 filas afectadas en el dataset sintético.
- **Recomendación para datos reales:** Definir una tabla de mapeo o aplicar normalización Unicode (NFD) para eliminar acentos de forma general.

In [27]:
# Corrección de valores conocidos (mapeo)
CORRECTIONS = {
    'flow_name': {
        'Gestión de Cuenta': 'Gestion de Cuenta',  # quitar acento
        'Gestion de Cuenta': 'Gestion de Cuenta'   # ya correcto
        # Añadir más si aparecen
    },
    'intencion': {
        # Podría venir 'problema pago' en lugar de 'problema_pago', etc.
    }
}

def normalize_categorical(df, corrections):
    """
    Aplica mapeos de corrección a columnas categóricas.
    Registra los cambios en un log.
    """
    change_log = {}
    for col, mapping in corrections.items():
        if col not in df.columns:
            continue
        # Contar cuántas filas cambian
        before = df[col].copy()
        df[col] = df[col].replace(mapping)
        changed = (before != df[col]).sum()
        if changed > 0:
            change_log[col] = changed
    return change_log

# Ejecutar (justo después de la validación, antes de guardar)
change_log = normalize_categorical(df_limpio, CORRECTIONS)
if change_log:
    print("📝 Correcciones aplicadas:")
    for col, count in change_log.items():
        print(f"   - {col}: {count} valores cambiados")
    # Guardar en el reporte
    report['corrections'] = change_log

📝 Correcciones aplicadas:
   - flow_name: 4983 valores cambiados


### 6. Guardar DataFrame limpio para siguiente etapa

In [28]:
# Si quieres persistir el DataFrame limpio por si falla la conexión a BD


if df_limpio is not None:
    OUTPUT_PARQUET = "../data/interim/data_limpia.parquet"
    df_limpio.to_parquet(OUTPUT_PARQUET, index=False)
    print(f"Datos limpios guardados en {OUTPUT_PARQUET}")

Datos limpios guardados en ../data/interim/data_limpia.parquet


In [29]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
import matplotlib.pyplot as plt


def generate_quality_report(df, original_shape, report_metadata, output_format='html'):
    """
    Genera un reporte de calidad de datos completo.
    
    Parámetros:
    - df: DataFrame limpio (post-validación)
    - original_shape: tuple (filas, columnas) antes de limpieza
    - report_metadata: dict con info de archivo, encoding, etc.
    - output_format: 'html', 'json', 'markdown'
    
    Retorna:
    - dict con todas las métricas
    - guarda archivo en disco
    """
    
    report = {
        'metadata': report_metadata,
        'summary': {
            'rows_initial': original_shape[0],
            'columns_initial': original_shape[1],
            'rows_final': df.shape[0],
            'columns_final': df.shape[1],
            'rows_removed': original_shape[0] - df.shape[0],
            'removed_percent': round((original_shape[0] - df.shape[0]) / original_shape[0] * 100, 2) if original_shape[0] > 0 else 0
        },
        'column_quality': {},
        'duplicates': {},
        'frustration_stats': {},
        'date_stats': {},
        'sample_errors': {}
    }
    
    # 1. Calidad por columna
    for col in df.columns:
        col_info = {
            'dtype': str(df[col].dtype),
            'null_count': int(df[col].isna().sum()),
            'null_percent': round(df[col].isna().sum() / len(df) * 100, 2),
            'unique_values': int(df[col].nunique()),
            'top_value': str(df[col].mode()[0]) if not df[col].mode().empty else None,
            'top_freq': int(df[col].value_counts().iloc[0]) if len(df[col].value_counts()) > 0 else 0
        }
        # Para columnas numéricas
        if pd.api.types.is_numeric_dtype(df[col]):
            col_info['min'] = float(df[col].min()) if not df[col].isna().all() else None
            col_info['max'] = float(df[col].max()) if not df[col].isna().all() else None
            col_info['mean'] = float(df[col].mean()) if not df[col].isna().all() else None
        report['column_quality'][col] = col_info
    
    # 2. Duplicados
    report['duplicates']['full_rows'] = int(df.duplicated().sum())
    # Duplicados por clave (session_id + turn_number)
    if 'session_id' in df.columns and 'turn_number' in df.columns:
        dup_keys = df.duplicated(subset=['session_id', 'turn_number']).sum()
        report['duplicates']['by_session_turn'] = int(dup_keys)
        # Mostrar ejemplos de claves duplicadas
        if dup_keys > 0:
            dup_examples = df[df.duplicated(subset=['session_id', 'turn_number'], keep=False)]\
                            .groupby(['session_id', 'turn_number']).size().reset_index(name='count')\
                            .head(5).to_dict('records')
            report['sample_errors']['duplicated_keys'] = dup_examples
    
    # 3. Estadísticas de frustración
    if 'nivel_frustracion' in df.columns:
        frust_series = df['nivel_frustracion'].dropna()
        report['frustration_stats'] = {
            'count': len(frust_series),
            'mean': float(frust_series.mean()),
            'std': float(frust_series.std()),
            'min': int(frust_series.min()),
            'max': int(frust_series.max()),
            'distribution': {
                '0': int((frust_series == 0).sum()),
                '1': int((frust_series == 1).sum()),
                '2': int((frust_series == 2).sum())
            }
        }
    
    # 4. Estadísticas de fecha (si existe)
    if 'fecha' in df.columns and pd.api.types.is_datetime64_any_dtype(df['fecha']):
        fecha_series = df['fecha'].dropna()
        report['date_stats'] = {
            'min': fecha_series.min().isoformat() if not fecha_series.empty else None,
            'max': fecha_series.max().isoformat() if not fecha_series.empty else None,
            'null_count': df['fecha'].isna().sum(),
            'range_days': (fecha_series.max() - fecha_series.min()).days if not fecha_series.empty else None
        }
    
    # 5. Ejemplos de valores problemáticos (fechas inválidas, etc.)
    if 'fecha' in df.columns:
        bad_dates = df[df['fecha'].isna() & df['fecha_original'].notna()] if 'fecha_original' in df.columns else []
        if len(bad_dates) > 0:
            report['sample_errors']['bad_dates'] = bad_dates[['fecha_original']].head(3).to_dict('records')
    
    # 6. Exportar según formato
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_filename = f"../reports/quality/quality_report_{timestamp}"
    
    if output_format == 'json':
        filename = f"{base_filename}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        print(f"Reporte JSON guardado: {filename}")
    
    elif output_format == 'html':
        # Generar HTML simple con tablas
        html_content = f"""
        <html>
        <head><title>Reporte de Calidad - Conversa AI</title>
        <style>
            body {{ font-family: Arial; margin: 20px; }}
            table {{ border-collapse: collapse; width: 100%; margin-bottom: 20px; }}
            th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            th {{ background-color: #f2f2f2; }}
            .summary {{ background-color: #e6f7ff; }}
        </style>
        </head>
        <body>
        <h1>Reporte de Calidad de Datos</h1>
        <p>Generado: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</p>
        <h2>Resumen</h2>
        <table class="summary">
        """
        for k, v in report['summary'].items():
            html_content += f"<tr><th>{k}</th><td>{v}</td></tr>"
        html_content += "</table><h2>Calidad por Columna</h2><table>"
        html_content += "<tr><th>Columna</th><th>Tipo</th><th>Nulos (%)</th><th>Únicos</th><th>Top Valor</th></tr>"
        for col, stats in report['column_quality'].items():
            html_content += f"<tr><td>{col}</td><td>{stats['dtype']}</td><td>{stats['null_count']} ({stats['null_percent']}%)</td>"
            html_content += f"<td>{stats['unique_values']}</td><td>{stats['top_value']} ({stats['top_freq']})</td></tr>"
        html_content += "</table>"
        if report['frustration_stats']:
            html_content += "<h2>Estadísticas de Frustración</h2><table>"
            for k, v in report['frustration_stats'].items():
                html_content += f"<tr><th>{k}</th><td>{v}</td></tr>"
            html_content += "</table>"
        html_content += "</body></html>"
        filename = f"{base_filename}.html"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(html_content)
        print(f"Reporte HTML guardado: {filename}")
    
    elif output_format == 'markdown':
        filename = f"{base_filename}.md"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"# Reporte de Calidad - {timestamp}\n\n")
            f.write("## Resumen\n")
            for k, v in report['summary'].items():
                f.write(f"- **{k}**: {v}\n")
            f.write("\n## Calidad por Columna\n")
            f.write("| Columna | Tipo | Nulos | Únicos | Top Valor |\n")
            f.write("|---------|------|-------|--------|-----------|\n")
            for col, stats in report['column_quality'].items():
                f.write(f"| {col} | {stats['dtype']} | {stats['null_count']} ({stats['null_percent']}%) | {stats['unique_values']} | {stats['top_value']} ({stats['top_freq']}) |\n")
        print(f"Reporte Markdown guardado: {filename}")
    
    return report

### 7. Ejecutar Reporte

In [30]:
# Suponiendo que df_limpio y report ya existen de la celda anterior
if df_limpio is not None:
    # Guardar una copia de la columna fecha original por si acaso (para debug)
    if 'fecha' in df_limpio.columns:
        df_limpio['fecha_original'] = df_limpio['fecha'].copy()
    
    # Asegurar carpeta de salida existe para evitar FileNotFoundError
    reports_dir = Path("../reports/quality")
    reports_dir.mkdir(parents=True, exist_ok=True)

    quality_report_dict = generate_quality_report(
        df=df_limpio,
        original_shape=(report['rows_initial'], len(report['columns_initial'])),
        report_metadata=report.get('metadata', {}),
        output_format='html'   # o 'json', 'markdown'
    )
    
    
    # Mostrar en el notebook un resumen bonito
    from IPython.display import display, HTML, Markdown
    display(Markdown("### 📊 Resumen del Reporte de Calidad"))
    display(HTML(f"<b>Filas eliminadas:</b> {quality_report_dict['summary']['rows_removed']} ({quality_report_dict['summary']['removed_percent']}%)"))
    display(HTML(f"<b>Duplicados totales:</b> {quality_report_dict['duplicates']['full_rows']}"))
    if 'by_session_turn' in quality_report_dict['duplicates']:
        display(HTML(f"<b>Duplicados por (session_id, turn_number):</b> {quality_report_dict['duplicates']['by_session_turn']}"))
    if quality_report_dict['frustration_stats']:
        dist = quality_report_dict['frustration_stats']['distribution']
        display(HTML(f"<b>Distribución frustración:</b> 0:{dist['0']} | 1:{dist['1']} | 2:{dist['2']}"))
    
    # Mostrar primeros registros problemáticos si existen
    if 'bad_dates' in quality_report_dict['sample_errors']:
        display(Markdown("#### ⚠️ Ejemplos de fechas inválidas"))
        display(pd.DataFrame(quality_report_dict['sample_errors']['bad_dates']))
else:
    print("No se puede generar reporte porque df_limpio es Non")

Reporte HTML guardado: ../reports/quality/quality_report_20260517_120327.html


### 📊 Resumen del Reporte de Calidad